# Feature Engineering — Individual Household Electric Power Consumption

Este notebook define y justifica, sobre el dataset limpio (`data/interim/household_power_cleaned.parquet`), las transformaciones necesarias para construir un dataset supervisado listo para entrenar un modelo de forecasting: frecuencia de trabajo definitiva, tratamiento del missing remanente, variable objetivo, horizonte de predicción, y las features de calendario, lags y rolling windows.

Como en los notebooks anteriores, este es un notebook **exploratorio**: cada decisión se toma y se verifica aquí con datos reales, y al final se traslada a un script reproducible (`src/features/build_features.py`) que no depende de Jupyter, siguiendo la misma separación investigación/producción usada en `src/cleaning/clean.py`.

## 1. Carga de datos

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
INTERIM_FILE = PROJECT_ROOT / "data" / "interim" / "household_power_cleaned.parquet"

df = pd.read_parquet(INTERIM_FILE)

print(f"Filas: {df.shape[0]:,}")
print(f"Rango temporal: {df.index.min()} -> {df.index.max()}")
df.head()

Filas: 2,075,259
Rango temporal: 2006-12-16 17:24:00 -> 2010-11-26 21:02:00


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 2. Frecuencia de trabajo definitiva

El EDA temporal (`02_eda.ipynb`) dejó esta decisión pendiente para esta etapa, considerando conjuntamente los resultados exploratorios y los requerimientos del modelo. Con esa evidencia ya disponible, se decide ahora.

**Decisión: frecuencia horaria.** Razones:

1. El patrón intradía (picos de mañana y noche) es la señal más fuerte identificada en el EDA, y desaparece si se agrega a nivel diario.
2. La autocorrelación encontrada en el EDA muestra picos recurrentes cada 24 horas y sus múltiplos — una estructura que solo es representable trabajando a nivel horario.
3. Las unidades ya definidas en el plan del proyecto para lags (1,2,3,24,48,168) y rolling windows (3,6,24,168) están expresadas en horas (24 = 1 día, 168 = 1 semana), lo que presupone esta granularidad.
4. 34,589 observaciones horarias es un tamaño perfectamente manejable, muy superior en detalle a las 1,442 observaciones diarias.

In [2]:
hourly = df["Global_active_power"].resample("h").mean()

print(f"Observaciones horarias: {len(hourly):,}")
print(f"Horas sin ningun dato: {hourly.isna().sum()} ({hourly.isna().mean():.2%})")

Observaciones horarias: 34,589
Horas sin ningun dato: 421 (1.22%)


## 3. Tratamiento del missing remanente

Tanto la limpieza (`01_data_quality_cleaning.ipynb`) como el EDA (`02_eda.ipynb`) documentaron y difirieron deliberadamente el tratamiento de estas 421 horas sin dato hasta esta etapa, porque la estrategia correcta depende de la frecuencia elegida (ya decidida en la sección 2) y de cómo esas horas faltantes afectarían a los lags/rolling windows que se construyen a continuación.

Antes de decidir cómo imputar, se cuantifica el verdadero impacto de NO imputar: como el lag más largo es de 168 horas (1 semana), una sola hora faltante "contamina" con NaN a las siguientes 168 filas de features derivadas de ella.

In [3]:
missing_mask = hourly.isna()

# Una fila se contamina si CUALQUIER hora entre t-168 y t (inclusive) esta faltante
contaminated = missing_mask.rolling(169, min_periods=1).max().astype(bool)

print(f"Horas faltantes originales: {missing_mask.sum()} ({missing_mask.mean():.2%})")
print(f"Filas que quedarian con NaN en algun lag/rolling si NO se imputa antes: "
      f"{contaminated.sum():,} ({contaminated.mean():.2%})")

Horas faltantes originales: 421 (1.22%)
Filas que quedarian con NaN en algun lag/rolling si NO se imputa antes: 1,765 (5.10%)


**Hallazgo clave:** no imputar multiplicaría por más de 4 el volumen de filas afectadas (de 1.22% a 5.10%), porque el efecto de cada hora faltante se propaga hacia adelante a través de los lags y rolling windows. Esto justifica imputar **antes** de construir las features, en vez de dejar el missing y perder filas después.

### Estrategia de imputación

Dado que el EDA encontró autocorrelación fuerte en los ciclos diario (24h) y semanal (168h), se utiliza una imputación estacional basada exclusivamente en información pasada.

La estrategia se aplica en cascada:

1. Llenar con el valor de **168 horas antes** (mismo día de la semana y misma hora), aprovechando el patrón semanal identificado en la autocorrelación.
2. Si ese valor también falta, utilizar el valor de **24 horas antes** (misma hora del día anterior).
3. Si ninguno de los dos valores históricos está disponible, el dato permanece como `NaN` y posteriormente la fila se excluirá del dataset supervisado si afecta alguna feature necesaria.

No se utiliza interpolación temporal como mecanismo de respaldo, ya que podría incorporar indirectamente observaciones posteriores al instante analizado y generar riesgo de data leakage en un escenario real de pronóstico.

In [4]:
hourly_filled = hourly.copy()

hourly_filled = hourly_filled.fillna(hourly.shift(168))
filled_by_168h = missing_mask & hourly_filled.notna()

hourly_filled = hourly_filled.fillna(hourly.shift(24))
filled_by_24h = missing_mask & ~filled_by_168h & hourly_filled.notna()

print(f"Horas rellenadas con el valor de 168h antes: {filled_by_168h.sum()}")
print(f"Horas rellenadas con el valor de 24h antes: {filled_by_24h.sum()}")
print(f"Missing remanente tras la imputacion causal: {hourly_filled.isna().sum()}")

Horas rellenadas con el valor de 168h antes: 421
Horas rellenadas con el valor de 24h antes: 0
Missing remanente tras la imputacion causal: 0


En este dataset, las 421 horas faltantes se resolvieron completamente utilizando el valor de 168 horas antes. Por lo tanto, el mecanismo de respaldo de 24 horas no fue necesario y no quedaron valores faltantes remanentes en la serie utilizada para construir las features.

Aunque en este conjunto de datos ambas estrategias de respaldo resultan suficientes, el pipeline conserva únicamente métodos basados en información pasada. No se utiliza interpolación temporal como último recurso, evitando que observaciones posteriores puedan influir en features utilizadas para predecir el futuro.

## 4. Variable objetivo y horizonte de predicción

**Target: `Global_active_power`.** Es la variable de negocio central del proyecto: el consumo activo total del hogar.

**Horizonte: 24 horas.** Se define el target como el valor de `Global_active_power` 24 horas en el futuro respecto a cada fila. Es la elección natural dado el ciclo diario dominante encontrado en el EDA (autocorrelación fuerte en el lag 24h) y corresponde a un caso de uso realista: pronosticar el consumo de "mañana a esta hora" con la información disponible hoy.

El target se construye a partir de la serie horaria original antes de la imputación. De esta manera, el modelo solo será entrenado con etiquetas correspondientes a observaciones reales. Si el valor ubicado 24 horas en el futuro originalmente está ausente, esa fila conservará un `NaN` en el target y será excluida posteriormente del dataset supervisado.

In [5]:
HORIZON_HOURS = 24

# El target se construye a partir de la serie horaria observada,
# antes de imputación, para evitar etiquetas sintéticas.
target = hourly.shift(-HORIZON_HOURS)

sample_idx = 1000
sample_time = hourly.index[sample_idx]
future_time = sample_time + pd.to_timedelta(HORIZON_HOURS, unit="h")

print(
    f"Ejemplo: target en {sample_time} = "
    f"{target.iloc[sample_idx]:.4f}"
)

print(
    f"         valor observado 24h después ({future_time}) = "
    f"{hourly.iloc[sample_idx + HORIZON_HOURS]:.4f}"
)

print(
    f"         coinciden: "
    f"{target.iloc[sample_idx] == hourly.iloc[sample_idx + HORIZON_HOURS]}"
)

Ejemplo: target en 2007-01-27 09:00:00 = 2.1713
         valor observado 24h después (2007-01-28 09:00:00) = 2.1713
         coinciden: True


## 5. Features de calendario

In [6]:
features = pd.DataFrame(index=hourly_filled.index)

features["hour"] = features.index.hour
features["day_of_week"] = features.index.dayofweek
features["month"] = features.index.month
features["is_weekend"] = (features.index.dayofweek >= 5).astype(int)

# Codificacion ciclica de la hora: sin/cos evitan la discontinuidad artificial
# entre 23h y 0h que tendria usar la hora como numero entero (23 -> 0 no es
# un salto pequeno para el modelo si no se codifica ciclicamente).
features["hour_sin"] = np.sin(2 * np.pi * features["hour"] / 24)
features["hour_cos"] = np.cos(2 * np.pi * features["hour"] / 24)

features.head()

,hour,day_of_week,month,is_weekend,hour_sin,hour_cos
datetime,,,,,,
2006-12-16 17:00:00,17,5,12,1,-0.965926,-2.588190e-01
2006-12-16 18:00:00,18,5,12,1,-1.000000,-1.836970e-16
2006-12-16 19:00:00,19,5,12,1,-0.965926,2.588190e-01
2006-12-16 20:00:00,20,5,12,1,-0.866025,5.000000e-01
2006-12-16 21:00:00,21,5,12,1,-0.707107,7.071068e-01


Estas cinco variables están directamente respaldadas por los patrones cuantificados en el EDA: `hour`/`hour_sin`/`hour_cos` por el patrón intradía (sección 5.1 de `02_eda.ipynb`), `day_of_week`/`is_weekend` por el patrón semanal (sección 5.2), y `month` por la estacionalidad anual (sección 5.3 y 6.2).

## 6. Features de lag

Un **lag** (rezago) es, simplemente, el valor de la propia serie en un momento anterior, "traído" a la fila actual como si fuera una columna más. Por ejemplo, `lag_1h` en la fila de las 14:00 es el valor real que tuvo `Global_active_power` a las 13:00; `lag_24h` en esa misma fila es el valor que tuvo exactamente un día antes, también a las 14:00.

¿Por qué esto es útil para un modelo de forecasting? Porque un modelo de Machine Learning "clásico" (regresión, árboles, etc.) no entiende de series de tiempo por sí solo — no sabe que una fila viene después de otra. Los lags son la forma de traducir "el pasado reciente" en columnas normales que el modelo sí puede usar como cualquier otra variable numérica. En la práctica, son la feature más importante en casi cualquier problema de forecasting: si el consumo de hace una hora fue alto, es muy probable que el de ahora también lo sea (autocorrelación).

La operación que construye un lag es `serie.shift(N)`: desplaza toda la serie N posiciones hacia adelante en el tiempo, de forma que el valor que "aparece" en la fila `t` es en realidad el valor original de la fila `t-N`. Por construcción, `shift()` nunca puede traer información del futuro — solo mueve valores que ya existían hacia atrás en el pasado, hacia adelante en la tabla. Esto se verifica explícitamente más adelante (sección 8).

In [7]:
LAGS_HOURS = [1, 2, 3, 24, 48, 168]

for lag in LAGS_HOURS:
    features[f"lag_{lag}h"] = hourly_filled.shift(lag)

features[[f"lag_{lag}h" for lag in LAGS_HOURS]].head(3)

,lag_1h,lag_2h,lag_3h,lag_24h,lag_48h,lag_168h
datetime,,,,,,
2006-12-16 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2006-12-16 18:00:00,4.222889,NaN,NaN,NaN,NaN,NaN
2006-12-16 19:00:00,3.632200,4.222889,NaN,NaN,NaN,NaN


Los seis lags corresponden directamente a los picos de autocorrelación encontrados en el EDA (sección 6.3 de `02_eda.ipynb`), no son valores elegidos arbitrariamente:

- **`lag_1h`, `lag_2h`, `lag_3h`** — dependencia de corto plazo: el consumo de una hora se parece mucho al de la hora inmediatamente anterior (autocorrelación >0.7 en el lag 1h) y esa relación decae rápido a medida que pasan las horas.
- **`lag_24h`** — ciclo diario: "¿cuánto se consumió a esta misma hora, ayer?". Es el segundo repunte de autocorrelación encontrado en el EDA, después de que la correlación de corto plazo ya se había disipado.
- **`lag_48h`** — dos días atrás; funciona como una referencia intermedia entre el ciclo diario y el semanal.
- **`lag_168h`** — ciclo semanal (168 = 24 × 7): "¿cuánto se consumió a esta misma hora, el mismo día de la semana pasada?". Es especialmente valioso porque captura el patrón de fin de semana (sección 5.2) sin necesidad de que el modelo lo aprenda desde cero.

Nótese que las primeras filas del dataset no tienen valor para `lag_168h` (ni para los demás lags): no existe "una semana antes" para las primeras 168 horas del historial. Esto se resuelve al final del notebook (sección 9), eliminando esas filas — es la razón principal por la que se pierden filas en el dataset final, no un error.

## 7. Rolling means y rolling std

Mientras que un lag mira **un único instante** del pasado (por ejemplo, "hace exactamente 24 horas"), una **rolling window** (ventana móvil) resume **un rango** de horas pasadas en un solo número. `rollmean_3h` en una fila es el promedio de las 3 horas anteriores; `rollstd_3h` es qué tan dispersos estuvieron esos 3 valores entre sí (la desviación estándar).

¿Por qué agregar esto si ya existen los lags? Porque capturan información distinta y complementaria:

- **Un lag individual puede ser ruidoso.** Un solo valor (por ejemplo `lag_1h`) puede estar afectado por un pico o caída puntual que no representa el comportamiento típico reciente. El **rolling mean** suaviza ese ruido, mostrando la tendencia de corto plazo en vez de un único punto.
- **El rolling std aporta información que ningún lag individual puede dar**: qué tan estable o volátil estuvo el consumo recientemente. Un hogar con consumo muy variable en las últimas horas (std alto) se comporta de forma distinta a uno con consumo estable (std bajo), incluso si ambos tienen el mismo promedio.

Se calculan cuatro ventanas — 3h, 6h, 24h y 168h — para capturar tendencia de muy corto plazo, medio día, un día completo y una semana completa, en la misma lógica que los lags de la sección anterior.

**Sobre el `shift(1)` — el punto más importante de esta sección para evitar leakage:** por defecto, una ventana `rolling(3)` calculada directamente sobre la serie en el instante `t` incluiría el propio valor de `t` (los 3 valores serían `t`, `t-1`, `t-2`). Eso sería una fuga de datos: en el momento real de predecir, todavía no se conoce el consumo de la hora actual — es justamente lo que, indirectamente, se está tratando de estimar. Por eso, antes de aplicar `rolling()`, la serie se desplaza un paso hacia atrás con `shift(1)`: así, la ventana de 3 horas en la fila `t` termina resumiendo `t-1`, `t-2` y `t-3` — estrictamente pasado, nunca el presente.

In [8]:
ROLLING_WINDOWS_HOURS = [3, 6, 24, 168]

# IMPORTANTE (leakage): se calcula sobre la serie desplazada un paso
# (shift(1)) antes de aplicar rolling, para que la ventana en el instante t
# resuma HORAS ANTERIORES a t y nunca incluya el propio valor de t.
shifted_for_rolling = hourly_filled.shift(1)

for w in ROLLING_WINDOWS_HOURS:
    features[f"rollmean_{w}h"] = shifted_for_rolling.rolling(w).mean()
    features[f"rollstd_{w}h"] = shifted_for_rolling.rolling(w).std()

features[[f"rollmean_{w}h" for w in ROLLING_WINDOWS_HOURS]].head(3)

,rollmean_3h,rollmean_6h,rollmean_24h,rollmean_168h
datetime,,,,
2006-12-16 17:00:00,NaN,NaN,NaN,NaN
2006-12-16 18:00:00,NaN,NaN,NaN,NaN
2006-12-16 19:00:00,NaN,NaN,NaN,NaN


In [9]:
# Verificacion explicita: el rolling mean de 3h en el instante t debe ser
# el promedio de t-1, t-2, t-3 (NUNCA t).
sample_t = features.index[500]
manual_mean = hourly_filled.loc[[sample_t - pd.to_timedelta(h, unit="h") for h in [1, 2, 3]]].mean()
print(f"rollmean_3h calculado en {sample_t}: {features.loc[sample_t, 'rollmean_3h']:.4f}")
print(f"promedio manual de t-1, t-2, t-3:        {manual_mean:.4f}")
print(f"coinciden: {np.isclose(features.loc[sample_t, 'rollmean_3h'], manual_mean)}")

rollmean_3h calculado en 2007-01-06 13:00:00: 2.7069
promedio manual de t-1, t-2, t-3:        2.7069
coinciden: True


La verificación anterior confirma exactamente ese comportamiento: el `rollmean_3h` calculado por el código coincide con el promedio manual de `t-1`, `t-2` y `t-3` — no con `t`, `t-1`, `t-2`. Esta comprobación puntual se repite de forma más sistemática y automatizada en la sección 8.

## 8. Verificación de que no se usa información futura (data leakage)

**¿Qué es exactamente el data leakage y por qué es peligroso?** Es cuando, sin darse cuenta, una feature usada para entrenar el modelo contiene información que en producción, en el momento real de hacer la predicción, todavía no existiría. El peligro no es solo teórico: un modelo con leakage suele mostrar métricas de validación excelentes (porque "hace trampa" viendo el futuro durante el entrenamiento) y luego falla en producción, donde ese futuro obviamente no está disponible. Es uno de los errores más comunes — y más difíciles de detectar a simple vista — en proyectos de forecasting, precisamente porque el código puede *verse* correcto y aun así filtrar información.

Antes de ensamblar el dataset final, se verifica explícitamente que ninguna feature construida usa información no disponible en el momento de la predicción. Esto es exactamente lo que pide el checklist del proyecto como tarea explícita del Día 6, no una simple revisión visual del código.

Se implementan tres tipos de chequeo, cada uno apuntando a una forma distinta en la que el leakage podría colarse:

1. **¿Cada `lag_Nh` realmente mira N horas hacia atrás?** Se compara, para una fila de muestra, el valor que produjo el código contra el valor real tomado manualmente N horas antes. Si no coincidieran, significaría que el `shift()` está mal aplicado (por ejemplo, con signo invertido).
2. **¿El `target` realmente mira 24 horas hacia adelante?** Aquí SÍ se espera mirar al futuro — es lo que se va a predecir — pero se confirma que sea exactamente el horizonte definido (24h) y no, por accidente, el valor actual o uno más cercano.
3. **¿Alguna feature coincide sospechosamente con el valor del instante actual `t`?** Este es el chequeo más general: si una columna que debería representar el pasado resultara ser casi siempre idéntica al valor contemporáneo de la serie, sería la señal más clara de que, en algún punto del código, se filtró el valor de `t` hacia una feature que se supone que no debería conocerlo.

In [10]:
checks = []

# 1) Cada lag_Nh en el instante t debe ser igual al valor real N horas antes.
sample_t = features.index[2000]
for lag in LAGS_HOURS:
    expected = hourly_filled.loc[sample_t - pd.Timedelta(lag, unit="h")]
    actual = features.loc[sample_t, f"lag_{lag}h"]
    checks.append((f"lag_{lag}h usa t-{lag}h (no el presente ni el futuro)", np.isclose(actual, expected)))

# 2) El target en t debe ser igual al valor real HORIZON_HOURS despues, nunca antes o en t.
expected_target = hourly.loc[
    sample_t + pd.to_timedelta(HORIZON_HOURS, unit="h")
]

checks.append((
    "target usa t+24h observado (el futuro que se predice)",
    np.isclose(target.loc[sample_t], expected_target)
))

# 3) Ninguna columna de features (excepto target) debe coincidir con el valor
#    contemporaneo de la propia serie objetivo en mas del 1% de las filas --
#    eso indicaria que se colo el valor de "t" en alguna feature. Se excluyen
#    explicitamente las filas que fueron IMPUTADAS con el valor de 168h antes
#    (seccion 3): para esas filas, filled[t] == filled[t-168] es un efecto
#    esperado y correcto de la imputacion, no una fuga de datos.
contemporaneous = hourly_filled.loc[features.index]
imputed_via_168h = filled_by_168h.reindex(features.index, fill_value=False)

for col in [c for c in features.columns if c.startswith(("lag_", "rollmean_", "rollstd_"))]:
    is_lag_168 = col == "lag_168h"
    mask = ~imputed_via_168h if is_lag_168 else pd.Series(True, index=features.index)
    exact_match = (features.loc[mask, col] == contemporaneous.loc[mask]).mean()
    label = f"{col} no es identico al valor contemporaneo de t"
    if is_lag_168:
        label += " (excluyendo las filas imputadas con el valor de 168h antes)"
    checks.append((label, exact_match < 0.01))

all_passed = all(ok for _, ok in checks)
for description, ok in checks:
    print(f"[{'OK' if ok else 'FALLO'}] {description}")

print()
print("Todas las verificaciones de leakage pasaron:" if all_passed else "HAY PROBLEMAS DE LEAKAGE:", all_passed)

[OK] lag_1h usa t-1h (no el presente ni el futuro)
[OK] lag_2h usa t-2h (no el presente ni el futuro)
[OK] lag_3h usa t-3h (no el presente ni el futuro)
[OK] lag_24h usa t-24h (no el presente ni el futuro)
[OK] lag_48h usa t-48h (no el presente ni el futuro)
[OK] lag_168h usa t-168h (no el presente ni el futuro)
[OK] target usa t+24h observado (el futuro que se predice)
[OK] lag_1h no es identico al valor contemporaneo de t
[OK] lag_2h no es identico al valor contemporaneo de t
[OK] lag_3h no es identico al valor contemporaneo de t
[OK] lag_24h no es identico al valor contemporaneo de t
[OK] lag_48h no es identico al valor contemporaneo de t
[OK] lag_168h no es identico al valor contemporaneo de t (excluyendo las filas imputadas con el valor de 168h antes)
[OK] rollmean_3h no es identico al valor contemporaneo de t
[OK] rollstd_3h no es identico al valor contemporaneo de t
[OK] rollmean_6h no es identico al valor contemporaneo de t
[OK] rollstd_6h no es identico al valor contemporaneo 

Las tres verificaciones confirman que: (1) cada lag efectivamente mira hacia atrás la cantidad exacta de horas que su nombre indica; (2) el target apunta exactamente 24 horas hacia adelante y corresponde a una observación real de la serie original, no a un valor imputado; y (3) ninguna feature de lag/rolling coincide de forma sospechosa con el valor contemporáneo de la serie, lo que reduce el riesgo de que se haya filtrado información del instante `t` hacia las features.

**Nota sobre `lag_168h`:** en una primera versión de este chequeo, `lag_168h` marcó una coincidencia del 1.22% con el valor contemporáneo — exactamente 422 filas, prácticamente idéntico a las 421 horas imputadas en la sección 3. La causa no es leakage: esas son precisamente las filas donde `hourly_filled[t]` se imputó copiando `hourly_filled[t-168]` (la estrategia estacional de la sección 3), así que para esas filas específicas `filled[t] == filled[t-168]` es un resultado esperado de la imputación, no una fuga de información del futuro. El chequeo se corrigió para excluir esas filas imputadas al evaluar `lag_168h`, y con esa exclusión también pasa. Este hallazgo es un buen recordatorio de que un chequeo de leakage puede dar falsos positivos cuando el pipeline incluye pasos previos (como la imputación) que crean coincidencias legítimas — por eso cada resultado inesperado se investigó antes de descartarlo, en vez de ignorarlo o borrarlo.

## 9. Ensamblado del dataset supervisado final

Hasta aquí, `features` es una tabla con columnas de calendario, lags, rolling stats y el target, pero varias filas todavía tienen huecos: las primeras 168 horas no tienen historia suficiente para calcular `lag_168h` (ni `rollmean_168h`), y las últimas 24 horas no tienen un valor real de `target` porque no existe información 24 horas después de las últimas observaciones del dataset. Estas dos situaciones son la única razón por la que se eliminan filas en este paso — el proyecto advierte explícitamente contra usar `dropna()` sin analizar antes su impacto, y aquí sí se analiza: no es una limpieza genérica, es la eliminación puntual y justificada de los bordes del dataset donde una fila, por definición, no puede tener toda la información necesaria.

In [11]:
# Agregamos el target observado al conjunto de features
features["target"] = target

print(
    f"Filas antes de eliminar historia insuficiente / "
    f"horizonte o target observado no disponible: {len(features):,}"
)

# Eliminamos únicamente las filas que no tienen toda la información
# necesaria para formar una observación supervisada válida
dataset = features.dropna()

print(f"Filas del dataset supervisado final: {len(dataset):,}")
print(f"Filas eliminadas: {len(features) - len(dataset):,}")

print("Desglose esperado:")
print("  168 filas iniciales sin historia suficiente para lag/rolling de 168h")
print("   24 filas finales sin horizonte disponible a t+24h")
print("  421 filas cuyo target t+24h corresponde a una observación originalmente faltante")
print("  Total esperado: 613 filas")

print(
    f"\nRango temporal final: "
    f"{dataset.index.min()} -> {dataset.index.max()}"
)

dataset.head(3)

Filas antes de eliminar historia insuficiente / horizonte o target observado no disponible: 34,589
Filas del dataset supervisado final: 33,976
Filas eliminadas: 613
Desglose esperado:
  168 filas iniciales sin historia suficiente para lag/rolling de 168h
   24 filas finales sin horizonte disponible a t+24h
  421 filas cuyo target t+24h corresponde a una observación originalmente faltante
  Total esperado: 613 filas

Rango temporal final: 2006-12-23 17:00:00 -> 2010-11-25 21:00:00


,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,lag_1h,lag_2h,lag_3h,lag_24h,...,lag_168h,rollmean_3h,rollstd_3h,rollmean_6h,rollstd_6h,rollmean_24h,rollstd_24h,rollmean_168h,rollstd_168h,target
datetime,,,,,,,,,,,,,,,,,,,,,
2006-12-23 17:00:00,17,5,12,1,-0.965926,-2.588190e-01,4.349100,4.049100,3.757967,1.496800,...,4.222889,4.052056,0.295578,3.608378,0.542196,2.934890,0.990187,1.763888,1.145995,1.686500
2006-12-23 18:00:00,18,5,12,1,-1.000000,-1.836970e-16,5.452533,4.349100,4.049100,2.686967,...,3.632200,4.616911,0.739052,3.990633,0.870877,3.099713,1.066674,1.771207,1.165554,0.505200
2006-12-23 19:00:00,19,5,12,1,-0.965926,2.588190e-01,3.879400,5.452533,4.349100,3.938167,...,3.400233,4.560344,0.807561,4.149067,0.710832,3.149397,1.074356,1.772679,1.168071,0.454033


Se eliminan únicamente las filas del principio (sin suficiente historia para calcular `lag_168h`) y del final (sin dato real de `target`, porque no existe información 24 horas después de las últimas observaciones del dataset). No se elimina ninguna fila por el missing tratado en la sección 3: ese problema ya se resolvió antes de construir las features, evitando la pérdida de las 1,765 filas que se hubieran contaminado (sección 3).

## 10. Guardado del dataset procesado

El resultado se guarda en `data/processed/`, no en `data/interim/` — esa distinción de carpetas refleja el rol de cada archivo dentro del pipeline: `data/interim/` contiene el dataset limpio pero todavía "crudo" en el sentido de que no tiene features de modelado (es el punto de partida común para cualquier análisis futuro), mientras que `data/processed/` contiene el dataset ya transformado en un problema de aprendizaje supervisado — con `target` incluido — listo para entrenar directamente. Al igual que en `01_data_quality_cleaning.ipynb`, se usa Parquet en vez de CSV porque preserva los tipos de datos (`float64`, índice `datetime`) y es más compacto y rápido de leer para los reentrenamientos futuros del pipeline (Día 7 en adelante).

In [12]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_FILE = PROCESSED_DIR / "features_hourly.parquet"
dataset.to_parquet(PROCESSED_FILE)

print(f"Dataset guardado en: {PROCESSED_FILE}")
print(f"Tamano del archivo: {PROCESSED_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Columnas: {dataset.columns.tolist()}")

Dataset guardado en: c:\Users\breid\Projects\Household-Power-Mlops\data\processed\features_hourly.parquet
Tamano del archivo: 4.47 MB
Columnas: ['hour', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h', 'rollmean_3h', 'rollstd_3h', 'rollmean_6h', 'rollstd_6h', 'rollmean_24h', 'rollstd_24h', 'rollmean_168h', 'rollstd_168h', 'target']


## 11. Conclusiones y resumen de decisiones

| Decisión | Elección | Justificación |
|---|---|---|
| Frecuencia | Horaria | Conserva el patrón intradía; unidades de lags/rolling del proyecto ya asumen horas (sección 2) |
| Tratamiento de missing |Imputación estacional causal en cascada (168h → 24h), utilizando únicamente información pasada. Si el valor continúa ausente, se conserva como NaN y la fila se excluye posteriormente si afecta una feature necesaria. | Evita la propagación de valores faltantes al construir lags/rolling sin introducir información futura. En este dataset, las 421 horas faltantes fueron resueltas utilizando el valor de 168h antes. |
| Target | `Global_active_power` | Variable de negocio central |
| Horizonte | 24 horas | Ciclo diario dominante (autocorrelación fuerte en lag 24h, EDA sección 6.3); caso de uso realista de pronóstico "a esta hora mañana" |
| Features de calendario | `hour`, `day_of_week`, `month`, `is_weekend`, `hour_sin`, `hour_cos` | Cada una respaldada por un patrón cuantificado en el EDA |
| Lags | 1, 2, 3, 24, 48, 168 horas | Coinciden con los picos de autocorrelación reales |
| Rolling mean/std | 3, 6, 24, 168 horas, calculados sobre la serie desplazada (`shift(1)`) | Resumen de corto/medio/largo plazo, sin incluir el valor del instante actual |
| Verificación de leakage | Explícita, con 3 tipos de chequeo (sección 8) | Requisito del checklist del proyecto, no solo inspección visual |
| Dataset final | 33,976 filas × 20 features + target, en data/processed/features_hourly.parquet | Se eliminan 613 filas: 168 iniciales sin historial suficiente, 24 finales sin horizonte t+24h y 421 cuyo target corresponde a una observación originalmente faltante. De esta manera, el modelo se entrenará únicamente con targets observados reales. |

### Siguiente paso

Este notebook es exploratorio. La lógica aquí definida y verificada se traslada a `src/features/build_features.py`, un script reproducible que aplica exactamente estas mismas transformaciones sin depender de Jupyter, listo para ser consumido por la etapa de entrenamiento (Día 7).